# 04 · Reaction transition states

A TS has partial forming and breaking bonds a force field may not type. Pin the reacting core from an `.xyz`
with `fix=[atoms]`, or state its distances and angles from SMILES with `fix={...}`. Pass the reacting atoms
to the geometry gate as `frozen=`. xyzrender uses 1-based atom indices for `ts_bonds=` and labels.

In [ ]:
import rxembed as rx
import xyzrender
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")

## Frozen TS core from an `.xyz`

A real spirocyclization TS (the forming C–C is the reacting bond). Freeze three reacting atoms so the core position and orientation are grafted onto every pose while the rest is searched. The forming bond is drawn dashed with its held length; the TS-aware gate checks the periphery.

In [ ]:
name = "spiro-ts1"
rc = [13, 14, 15]  # three atoms fix the reacting-core position and orientation
bonds = [(14, 15)]  # forming / breaking bonds held during embed
path = f"structures/{name}.xyz"
ens = rx.embed(path, fix=rc, n=4).mc(preset="rapid").minimize().prune()
reports = ens.check(frozen=rc, reference=path)
print(f"{len(ens)} conformers; fixed core checked against the input XYZ")
for report in reports.values():
    report.assert_ok()
xyzrender.render(
    ens.mol,
    overlay=f"structures/{name}.xyz",
    align_atoms=[13, 14, 15],
    no_hy=True,
    ts_bonds=[(i + 1, j + 1) for i, j in bonds],
    labels=[f"{i + 1} {j + 1} d" for i, j in bonds],
)

### The geometry tier: `optimize` with the frozen core held

Embedding and search give seeds. `optimize()` relaxes the periphery with the grafted core held. The example
uses GFN-FF for dispersion and hydrogen-bond terms and requires `xtb` on `$XTB_EXE`.

In [ ]:
opt = ens.lowest(1).optimize("gfnff")  # GFN-FF geometry opt; the frozen core (from the graft) is held automatically
rep = opt.check(frozen=rc, reference=path)[opt.ids[0]]
rep.assert_ok()
print(f"GFN-FF optimize: {opt.n} conformer; fixed core retained")

## Constrained TS from SMILES: a 3-centre SN2

No geometry is needed: give target reacting-atom distances and a near-linear angle. A single distance becomes a ±0.05 Å restrained window; `.measure()` reports the realised geometry. The forming F···C and breaking C···Cl are the TS bonds.

In [ ]:
smi = "[F-].CCCCCl"
f, c, cl = rx.match(rx.parse_smiles(smi), "[F-].[CH2][Cl]")
# fix= (rigid) the forming/breaking lengths + the linear angle; verify with .measure() (a pull, not a hard snap)
sn2 = rx.embed(smi, fix={(f, c): 2.02, (c, cl): 2.28, (f, c, cl): 178}).mc(preset="ensemble").minimize().prune()
for report in sn2.check(frozen=(f, c, cl)).values():
    report.assert_ok()
print(
    f"{len(sn2)} SN2-guess conformers | F\u00b7\u00b7\u00b7C {sn2.measure((f, c))['mean']:.2f} \u00c5, "
    f"C\u00b7\u00b7\u00b7Cl {sn2.measure((c, cl))['mean']:.2f} \u00c5, "
    f"F-C-Cl {sn2.measure((f, c, cl))['mean']:.0f}\u00b0"
)
xyzrender.render(
    sn2.mol,
    no_hy=True,
    ts_bonds=[(f + 1, c + 1), (c + 1, cl + 1)],
    labels=[f"{f + 1} {c + 1} d", f"{c + 1} {cl + 1} d"],
)

## FLP borylation and the TS-aware gate

Five target distances bias a concerted B–H···C–H transfer. Numeric `fix=` does not create a frozen atom set, so the gate initially reads the forming B···C bond as a clash. Passing the reacting atoms as `frozen=` makes that check TS-aware.

In [ ]:
smi = "CB(C1=C(N(C)C)C=CC=C1).CN1C=CC=C1"  # aryl-borane + 1-methylpyrrole
mol = Chem.AddHs(Chem.MolFromSmiles(smi))
(B,) = rx.match(mol, "[#5]")
(aN,) = rx.match(mol, "[N;!a]")
pN, C = mol.GetSubstructMatch(Chem.MolFromSmarts("[nX3][cH]"))  # either alpha-C is equivalent here
H = next(h.GetIdx() for h in mol.GetAtomWithIdx(C).GetNeighbors() if h.GetAtomicNum() == 1)
react = [B, H, C, aN, pN]
core = {(B, H): 2.08, (aN, H): 1.51, (C, H): 1.29, (B, C): 1.69, (B, aN): 3.06}

flp = rx.embed(smi, fix=core).mc(preset="ensemble").minimize().prune()
print(f"{len(flp)} FLP-borylation TS conformers")
# verify directly: a from-SMILES fix= biases the seed, so a tight forming-bond target
# (B···C 1.69) is pulled toward but not snapped: the log's "constraint not held" is that bias falling short;
# a quantum TS optimisation should refine it. measure() shows what actually embedded.
print("reacting-core distances (target -> realised mean):")
for (i, j), tgt in core.items():
    print(f"  d({i},{j}): {tgt:.2f} -> {flp.measure((i, j))['mean']:.2f} Å")
print("gate WITHOUT frozen (forming B···C looks like a clash):")
lo = flp.lowest(1)
rep = lo.check()[lo.ids[0]]
print("without TS exclusions:", "clean" if rep.ok() else rep.violations[0])
total = flp.n
flp.filter("geometry", frozen=react)
print(f"TS-aware gate: {flp.n}/{total} clean")
ts = [(B, H), (aN, H), (C, H)]  # the three transferring-H bonds
xyzrender.render(
    flp.mol, no_hy=True, ts_bonds=[(i + 1, j + 1) for i, j in ts], labels=[f"{i + 1} {j + 1} d" for i, j in ts]
)

## A TS with multiple non-covalent grips

A bifunctional iminophosphorane–thiourea TS (172 atoms, 3 fragments). `fix=` holds the reacting core exactly while rxembed discovers the distinct NCI binding modes at the TS geometry, the catalyst gripping the substrate several ways, and conf-searches each. The seeded contacts are drawn with `nci_bonds=` (dotted), distinct from the frozen reacting core.

In [ ]:
name = "bimp"
rc = [10, 11, 12, 14]  # reacting-core atoms (from a graphrc trajectory in practice)
modes = rx.nci_modes(rx.read_xyz(f"structures/{name}.xyz", 0))  # binding modes at the frozen TS geometry
print(f"{len(modes)} NCI binding modes discovered at the bimp TS; searching the first two (core frozen):")
best, shown = None, 0
for label, mode in modes.items():
    ens = rx.embed(f"structures/{name}.xyz", fix=rc, contacts=mode, n=8).mc(preset="rapid").minimize().prune()
    if not ens.ids:
        continue
    shown += 1
    print(f"  grip {shown}: {len(ens):2d} confs | {label[:46]}")
    best = best or (ens, mode)
    if shown >= 2:
        break
ens, mode = best
ens.filter("geometry", frozen=rc, reference=f"structures/{name}.xyz")
grip = [(min(k) + 1, max(k) + 1) for k in mode.distances]  # the seeded H-bond contacts
xyzrender.render(ens.mol, no_hy=True, nci_bonds=grip)